In [5]:
TRAIN_LABELS = "/content/drive/MyDrive/Major_Project/10_class/images/test"
VAL_LABELS  = "/content/drive/MyDrive/Major_Project/10_class/labels/test"

!ls "$TRAIN_LABELS" | wc -l
!ls "$VAL_LABELS" | wc -l

1945
1945


In [3]:
import os
import hashlib
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

IMAGES_FOLDER = "/content/drive/MyDrive/Major_Project/10_class/images/test"
LABELS_FOLDER = "/content/drive/MyDrive/Major_Project/10_class/labels/test"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


# ─────────────────────────────────────────────
# STEP 1: GET ALL FILES
# ─────────────────────────────────────────────

image_files = sorted([
    f for f in os.listdir(IMAGES_FOLDER)
    if Path(f).suffix.lower() in IMAGE_EXTENSIONS
])

label_files = sorted([
    f for f in os.listdir(LABELS_FOLDER)
    if Path(f).suffix.lower() == ".txt"
])

print("=" * 60)
print("📂  BEFORE CLEANING")
print("=" * 60)
print(f"  Total Images : {len(image_files)}")
print(f"  Total Labels : {len(label_files)}")
print("=" * 60)


# ─────────────────────────────────────────────
# STEP 2: FIND DUPLICATE IMAGES BY FILE HASH
# ─────────────────────────────────────────────
# Two images are duplicates if their file content is identical.
# We keep the first occurrence and delete the rest.

def file_hash(filepath):
    """MD5 hash of file content — identical files → identical hash."""
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


print("\n🔍  Scanning for duplicate images by file content hash...")

seen_hashes   = {}   # hash → first filename kept
duplicate_images = []  # filenames to delete

for img_name in image_files:
    img_path = os.path.join(IMAGES_FOLDER, img_name)
    h        = file_hash(img_path)

    if h in seen_hashes:
        duplicate_images.append(img_name)
        print(f"  ❌ Duplicate: {img_name}  (same as {seen_hashes[h]})")
    else:
        seen_hashes[h] = img_name

print(f"\n  Found {len(duplicate_images)} duplicate image(s)")


# ─────────────────────────────────────────────
# STEP 3: ALSO FIND ORPHAN LABELS
# (label .txt exists but no matching image)
# ─────────────────────────────────────────────

image_bases = {Path(f).stem for f in image_files
               if f not in duplicate_images}

orphan_labels = []
for lbl_name in label_files:
    base = Path(lbl_name).stem
    if base not in image_bases:
        orphan_labels.append(lbl_name)
        print(f"  ⚠️  Orphan label (no matching image): {lbl_name}")

print(f"\n  Found {len(orphan_labels)} orphan label(s)")


# ─────────────────────────────────────────────
# STEP 4: DELETE DUPLICATES + PAIRED LABELS
# ─────────────────────────────────────────────

deleted_images = 0
deleted_labels = 0

print("\n" + "=" * 60)
print("🗑️   DELETING DUPLICATES")
print("=" * 60)

for img_name in duplicate_images:
    img_path = os.path.join(IMAGES_FOLDER, img_name)
    base     = Path(img_name).stem
    lbl_path = os.path.join(LABELS_FOLDER, base + ".txt")

    # Delete image
    os.remove(img_path)
    deleted_images += 1
    print(f"  🗑  Deleted image : {img_name}")

    # Delete paired label if it exists
    if os.path.exists(lbl_path):
        os.remove(lbl_path)
        deleted_labels += 1
        print(f"  🗑  Deleted label : {base}.txt")

# Delete orphan labels
for lbl_name in orphan_labels:
    lbl_path = os.path.join(LABELS_FOLDER, lbl_name)
    os.remove(lbl_path)
    deleted_labels += 1
    print(f"  🗑  Deleted orphan label : {lbl_name}")

print(f"\n  Deleted {deleted_images} image(s)")
print(f"  Deleted {deleted_labels} label(s)")


# ─────────────────────────────────────────────
# STEP 5: FINAL COUNT
# ─────────────────────────────────────────────

final_images = sorted([
    f for f in os.listdir(IMAGES_FOLDER)
    if Path(f).suffix.lower() in IMAGE_EXTENSIONS
])

final_labels = sorted([
    f for f in os.listdir(LABELS_FOLDER)
    if Path(f).suffix.lower() == ".txt"
])

print("\n" + "=" * 60)
print("✅  AFTER CLEANING")
print("=" * 60)
print(f"  Total Images : {len(final_images)}")
print(f"  Total Labels : {len(final_labels)}")
print("=" * 60)


# ─────────────────────────────────────────────
# STEP 6: INTEGRITY CHECK
# (every image should have a label and vice versa)
# ─────────────────────────────────────────────

final_image_bases = {Path(f).stem for f in final_images}
final_label_bases = {Path(f).stem for f in final_labels}

missing_labels = final_image_bases - final_label_bases
missing_images = final_label_bases - final_image_bases

print("\n📋  INTEGRITY CHECK")
print("=" * 60)
if not missing_labels and not missing_images:
    print("  ✅  Every image has a label and every label has an image.")
else:
    if missing_labels:
        print(f"  ⚠️  {len(missing_labels)} image(s) have no label:")
        for name in sorted(missing_labels):
            print(f"       {name}")
    if missing_images:
        print(f"  ⚠️  {len(missing_images)} label(s) have no image:")
        for name in sorted(missing_images):
            print(f"       {name}")

print("=" * 60)

📂  BEFORE CLEANING
  Total Images : 1945
  Total Labels : 1957

🔍  Scanning for duplicate images by file content hash...

  Found 0 duplicate image(s)
  ⚠️  Orphan label (no matching image): CompLeft_g_130 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_132 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_133 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_136 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_137 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_138 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_139 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_14 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_140 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_141 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_142 (1).txt
  ⚠️  Orphan label (no matching image): CompLeft_g_143 (1).txt

  Found 12 orphan label(s)

🗑️   DELETING DUPLICATES
  🗑  Deleted orphan label : CompLeft_g_1